### MLflow setup:

tracking server: yes, local server : mlflow server
backend store: sqlite database : --backend-store-uri sqlite:///backend.db
artifacts store: local filesystem

In [15]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [16]:

print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://127.0.0.1:5000'


In [17]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1787672274806, experiment_id='1', last_update_time=1787672274806, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1787671951451, experiment_id='0', last_update_time=1787671951451, lifecycle_stage='active', name='Default', tags={}>]

In [18]:

from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models", input_example=X[:1])
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/08/25 15:38:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


default artifacts URI: 'mlflow-artifacts:/1/bcaccbcf52784d88833d9c508510dbaf/artifacts'
🏃 View run classy-donkey-864 at: http://127.0.0.1:5000/#/experiments/1/runs/bcaccbcf52784d88833d9c508510dbaf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [19]:

mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1787672274806, experiment_id='1', last_update_time=1787672274806, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1787671951451, experiment_id='0', last_update_time=1787671951451, lifecycle_stage='active', name='Default', tags={}>]

### Interacting with the model registry

In [20]:

from mlflow.tracking import MlflowClient


client = MlflowClient("http://127.0.0.1:5000")

In [21]:
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1787672279284, deployment_job_id='', deployment_job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', description='', last_updated_timestamp=1787672279414, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1787672279414, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1787672279414, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='83928ebda57148c286910f6b534e59ea', run_link='', source='models:/m-a92da987ec424bb9bfd8225ab7557e07', status='READY', status_message=None, tags={}, user_id='', version='1'>], name='iris-classifier', tags={}>]

In [14]:
experiment = client.get_experiment_by_name("my-experiment-1")
run_id = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["attributes.start_time DESC"],
    max_results=1,
)[0].info.run_id

mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name="iris-classifier",
)

Successfully registered model 'iris-classifier'.
2026/08/25 15:37:59 WARNING mlflow.tracking._model_registry.fluent: Run with id 83928ebda57148c286910f6b534e59ea has no artifacts at artifact path 'models', registering model based on models:/m-a92da987ec424bb9bfd8225ab7557e07 instead
2026/08/25 15:37:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1787672279414, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1787672279414, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='83928ebda57148c286910f6b534e59ea', run_link='', source='models:/m-a92da987ec424bb9bfd8225ab7557e07', status='READY', status_message=None, tags={}, user_id='', version='1'>